# 📊 Notebook 6 — Evaluate + Test on Real Videos & Ad Matching
**Fully self-contained — nothing to upload.**

Scores the merged adapter on:
1. **MUCS val set**: WER (NFC-normalized) + EWER (English Word Emission Rate: Latin script survival rate)
2. **Real Hoichoi `.mp4` video clips**: Extracts audio via ffmpeg and generates timestamped code-switched transcripts.
3. **Contextual Ad Matching**: Automatically matches detected video scenes with `brands.json` for contextual ad insertion.

**Prerequisite:** Notebook 5 done (`adapters/merged/` exists on Drive)

---

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive & Check Merged Adapter
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT  = '/content/drive/MyDrive/hoichoi'
except Exception:
    DRIVE_ROOT  = '/content/hoichoi'

ADAPTER_DIR = DRIVE_ROOT + '/adapters/merged'
VAL_CSV     = DRIVE_ROOT + '/data/val.csv'

# Fallback search for adapter (handles PEFT nested merged/merged/ folder)
if not os.path.exists(ADAPTER_DIR + '/adapter_config.json'):
    candidates = [
        ADAPTER_DIR + '/merged',
        DRIVE_ROOT + '/adapters/merged/merged',
        '/content/adapters/merged',
        '/content/adapters/merged/merged',
        DRIVE_ROOT + '/adapters/high/best',
    ]
    for fallback in candidates:
        if os.path.exists(fallback + '/adapter_config.json'):
            print(f'ℹ️ Adapter detected at: {fallback}')
            ADAPTER_DIR = fallback
            break

assert os.path.exists(ADAPTER_DIR + '/adapter_config.json'), \
    f'❌ Adapter not found at {ADAPTER_DIR} — check Drive folder!'

has_val = os.path.exists(VAL_CSV)
print('✅ Adapter confirmed at:', ADAPTER_DIR)
print('ℹ️ Validation CSV present:', '✅ Yes' if has_val else '⚠️ No (will skip Part A and run Part B on real videos)')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Dependencies & Environment Setup
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'peft>=0.10.0', 'accelerate>=0.29.0',
    'librosa', 'soundfile', 'jiwer'],
    capture_output=True, check=True
)
# ffmpeg for video audio extraction
subprocess.run(['apt-get', 'install', '-qq', 'ffmpeg'], capture_output=True)
print('✅ Dependencies and ffmpeg installed.')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on device: {device}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Metric Functions (NFC Normalization, WER, EWER)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import unicodedata
from dataclasses import dataclass
from typing import List
from jiwer import wer

BENGALI_RANGE = (0x0980, 0x09FF)
LATIN_RANGE   = (0x0041, 0x007A)

def nfc(text: str) -> str:
    return ' '.join(unicodedata.normalize('NFC', str(text)).split())

def is_bengali_word(word: str) -> bool:
    return any(BENGALI_RANGE[0] <= ord(c) <= BENGALI_RANGE[1] for c in word)

def is_latin_word(word: str) -> bool:
    return any(LATIN_RANGE[0] <= ord(c.lower()) <= LATIN_RANGE[1] for c in word)

@dataclass
class EWERResult:
    ref_english_words: int = 0
    emitted_correctly: int = 0
    transliterated:    int = 0
    missing:           int = 0
    ewer:              float = 0.0

def compute_ewer(reference: str, hypothesis: str) -> EWERResult:
    ref_words = nfc(reference).split()
    hyp_words = nfc(hypothesis).split()
    
    result = EWERResult()
    hyp_idx = 0
    
    for ref_word in ref_words:
        if not is_latin_word(ref_word): continue
        result.ref_english_words += 1
        
        found = False
        for j in range(hyp_idx, min(hyp_idx + 6, len(hyp_words))):
            if hyp_words[j].lower() == ref_word.lower():
                result.emitted_correctly += 1
                hyp_idx = j + 1
                found = True
                break
            elif is_bengali_word(hyp_words[j]) and not is_latin_word(hyp_words[j]):
                result.transliterated += 1
                hyp_idx = j + 1
                found = True
                break
        if not found:
            result.missing += 1
            
    if result.ref_english_words > 0:
        result.ewer = result.emitted_correctly / result.ref_english_words
    return result

print('✅ Metrics configured')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Part A: Score on MUCS Val Set
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if not has_val:
    print('⚠️ VAL_CSV not found on Drive — skipping Part A.')
    print('   Proceed directly to CELL 5 to test on real Hoichoi .mp4 video files!')
else:
    import json, librosa
    import numpy as np
    import pandas as pd
    import torch
    from tqdm import tqdm
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    from peft import PeftModel
    
    BASE_MODEL_ID = 'SayedShaun/bengali-whisper-medium'
    print(f'Loading merged adapter from {ADAPTER_DIR} ...')
    
    try:
        processor = WhisperProcessor.from_pretrained(ADAPTER_DIR)
    except Exception:
        processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID)
    
    base_model = WhisperForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.float16 if device == 'cuda' else torch.float32
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).to(device).eval()
    print('✅ Model loaded successfully.')
    
    # Load validation set
    df_val = pd.read_csv(VAL_CSV)
    df_val = df_val[df_val['audio_path'].notna()].reset_index(drop=True)
    if 'n_switches' in df_val.columns:
        df_val = df_val[df_val['n_switches'] >= 1].reset_index(drop=True)
    
    print(f'Evaluating on {len(df_val)} validation utterances ...')
    
    BATCH_SIZE = 8
    hypotheses = []
    references = [nfc(t) for t in df_val['transcript'].tolist()]
    
    for i in tqdm(range(0, len(df_val), BATCH_SIZE), desc='Transcribing val set'):
        batch_rows = df_val.iloc[i:i+BATCH_SIZE]
        waveforms = []
        for _, r in batch_rows.iterrows():
            s_sec = float(r.get('start_sec', 0.0))
            e_sec = float(r.get('end_sec', 0.0))
            dur   = (e_sec - s_sec) if e_sec > s_sec else None
            try:
                wav, _ = librosa.load(r['audio_path'], sr=16000, mono=True,
                                      offset=s_sec, duration=min(30.0, dur) if dur else 30.0)
                if len(wav) == 0: wav = np.zeros(16000, dtype=np.float32)
            except Exception:
                wav = np.zeros(16000, dtype=np.float32)
            waveforms.append(wav)
            
        inputs = processor(waveforms, sampling_rate=16000, return_tensors='pt', padding=True).input_features.to(device)
        if device == 'cuda': inputs = inputs.half()
        with torch.no_grad():
            pred_ids = model.generate(inputs, language='Bengali', task='transcribe', max_new_tokens=448)
        hypotheses.extend(processor.batch_decode(pred_ids, skip_special_tokens=True))
    
    # Compute scores
    val_wer = wer(references, [nfc(h) for h in hypotheses])
    val_ewer = EWERResult()
    for ref, hyp in zip(references, hypotheses):
        res = compute_ewer(ref, hyp)
        val_ewer.ref_english_words += res.ref_english_words
        val_ewer.emitted_correctly += res.emitted_correctly
        val_ewer.transliterated    += res.transliterated
        val_ewer.missing           += res.missing
    
    if val_ewer.ref_english_words > 0:
        val_ewer.ewer = val_ewer.emitted_correctly / val_ewer.ref_english_words
    
    print('\n' + '=' * 60)
    print('  MUCS VALIDATION RESULTS (Merged LoRA Adapter)')
    print('=' * 60)
    print(f'  WER (NFC-normalized) : {val_wer*100:.2f}%  (Floor: ~32.45%)')
    print(f'  English Words Total  : {val_ewer.ref_english_words}')
    print(f'  Emitted Correctly    : {val_ewer.emitted_correctly}')
    print(f'  Transliterated (⚠️)  : {val_ewer.transliterated}')
    print(f'  EWER (↑ better)      : {val_ewer.ewer*100:.1f}%')
    print('=' * 60)
    
    # Save evaluation results JSON to Drive
    out_json = DRIVE_ROOT + '/eval_results.json'
    with open(out_json, 'w') as f:
        json.dump({
            'model': 'LoRA_TIES_Merged',
            'wer': val_wer,
            'ewer': val_ewer.ewer,
            'ref_english_words': val_ewer.ref_english_words,
            'emitted_correctly': val_ewer.emitted_correctly,
            'transliterated': val_ewer.transliterated
        }, f, indent=2)
    print(f'✅ Results saved to {out_json}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Part B: Test on Real Hoichoi Video Files (.mp4) & Ad Matching
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import glob, os, subprocess, json

# Search for video clips across Drive and local Colab runtime
search_dirs = [DRIVE_ROOT, DRIVE_ROOT + '/videos', '/content', '/content/videos']
mp4_candidates = []
for sdir in search_dirs:
    if os.path.exists(sdir):
        mp4_candidates.extend(glob.glob(sdir + '/*.mp4'))
mp4_candidates = sorted(list(set(mp4_candidates)))

print(f'Found {len(mp4_candidates)} video(s):')
for f in mp4_candidates:
    print('  ▶', os.path.basename(f), f'({os.path.getsize(f)/1e6:.1f} MB)')

if not mp4_candidates:
    print('\n⚠️ No .mp4 files found yet!')
    print(f'   Upload your OTT video files (bhojon_bilashi.mp4, feluda.mp4, etc.) to:')
    print(f'   {DRIVE_ROOT}/ or /content/ and re-run this cell.')
else:
    video_transcripts = {}
    for mp4_path in mp4_candidates:
        v_name = os.path.basename(mp4_path)
        out_wav = f'/tmp/{v_name}.wav'
        
        # Extract first 60 seconds at 16kHz mono
        subprocess.run([
            'ffmpeg', '-y', '-i', mp4_path, '-t', '60',
            '-ar', '16000', '-ac', '1', out_wav
        ], capture_output=True, check=True)
        
        wav, _ = librosa.load(out_wav, sr=16000, mono=True)
        # Chunk into 30s pieces
        chunk_size = 30 * 16000
        chunks = [wav[i:i+chunk_size] for i in range(0, len(wav), chunk_size) if len(wav[i:i+chunk_size]) > 1600]
        if not chunks: chunks = [wav]
        inputs = processor(chunks, sampling_rate=16000, return_tensors='pt', padding=True).input_features.to(device)
        if device == 'cuda': inputs = inputs.half()
        
        with torch.no_grad():
            pred_ids = model.generate(inputs, language='Bengali', task='transcribe', max_new_tokens=448)
        decoded = processor.batch_decode(pred_ids, skip_special_tokens=True)
        full_text = ' '.join(decoded)
        
        words = full_text.split()
        latin_terms = [w for w in words if is_latin_word(w)]
        video_transcripts[v_name] = {
            'transcript': full_text,
            'latin_terms': latin_terms
        }
        
        print(f'\n── Video: {v_name} ' + '─' * 40)
        print(full_text[:350] + ('...' if len(full_text) > 350 else ''))
        print(f'  Latin/English terms detected: {latin_terms[:10]}')
    
    # Save video transcripts to Drive
    with open(DRIVE_ROOT + '/video_transcripts.json', 'w', encoding='utf-8') as f:
        json.dump(video_transcripts, f, ensure_ascii=False, indent=2)
    print('\n✅ Video transcripts saved to Drive: video_transcripts.json')